In [1]:
import fitz  # PyMuPDF
import pdfplumber
import os
import re

In [41]:
def extract_text_pymupdf(pdf_path):
    doc = fitz.open(pdf_path)
    text = ""
    for page in doc:
        text += page.get_text()
        
    return text





def extract_text_pymupdf_rawdict_safe(pdf_path, tab_gap_threshold=15):
    """
    Extract text from PDF using 'rawdict', safely handling missing 'text' keys.
    Preserves:
    - line breaks
    - paragraph breaks
    - simulated tabs for large gaps
    """
    doc = fitz.open(pdf_path)
    full_text = ""

    for page in doc:
        raw = page.get_text("rawdict")

        for block in raw["blocks"]:
            if block.get("type") != 0:  # skip non-text blocks
                continue

            for line in block.get("lines", []):
                line_text = ""
                last_x1 = None

                for span in line.get("spans", []):
                    # skip spans without text
                    if "text" not in span or not span["text"].strip():
                        continue

                    # calculate gap for tabs
                    if last_x1 is not None:
                        gap = span["bbox"][0] - last_x1
                        if gap > tab_gap_threshold:
                            line_text += "\t"
                        else:
                            line_text += " "
                    line_text += span["text"]
                    last_x1 = span["bbox"][2]

                if line_text.strip():  # only add non-empty lines
                    full_text += line_text + "\n"

        full_text += "\n"  # extra newline between pages

    return full_text






def extract_text_with_tabs_and_spaces(pdf_path, tab_threshold=50):
    """
    Extract text from PDF keeping line breaks, normal spaces between words,
    and converting large indentations into real tabs (\t).
    
    tab_threshold: horizontal distance (in PDF points) that counts as a tab.
    """
    final_text = ""

    with pdfplumber.open(pdf_path) as pdf:
        for page in pdf.pages:
            words = page.extract_words()
            lines = {}

            # Group words by vertical position
            for word in words:
                y_top = round(word['top'], 1)
                if y_top not in lines:
                    lines[y_top] = []
                lines[y_top].append(word)

            # Process lines top-to-bottom
            for y in sorted(lines.keys()):
                line_words = sorted(lines[y], key=lambda w: w['x0'])
                line_text = ""
                last_x = 0
                for i, w in enumerate(line_words):
                    # Calculate horizontal distance from last word
                    distance = w['x0'] - last_x
                    if i == 0:
                        # Leading spaces: convert to tabs if large
                        tab_count = int(distance / tab_threshold)
                        line_text += "\t" * tab_count
                    else:
                        # Between words: keep a single space
                        if distance > 2:  # small tolerance to avoid squishing
                            line_text += " "
                    line_text += w['text']
                    last_x = w['x1']
                final_text += line_text + "\n"

    return final_text










def extract_text_pdfplumber(pdf_path):
    text = ""
    with pdfplumber.open(pdf_path) as pdf:
        for page in pdf.pages:
            text += page.extract_text(x_tolerance=4, y_tolerance=2) + "\n"
    return text



def extract_text_pdfplumber_layout(pdf_path):
    text = ""
    with pdfplumber.open(pdf_path) as pdf:
        for page in pdf.pages:
            text += page.extract_text(layout=True) + "\n"
    return text

In [38]:
pdf_folder = 'PDFs'
txt_folder = 'TXTs_brut'
pdfs = sorted([
    pdf[:-4] for pdf in os.listdir(pdf_folder)
    if pdf.endswith(".pdf")
])


In [42]:
i=0
for pdf in pdfs :
    print(pdf)
    #text = extract_text_pymupdf(pdf_folder + "/" + pdf + ".pdf")
    text = extract_text_pdfplumber_layout(pdf_folder + "/" + pdf + ".pdf")
    #text = extract_text_pdfplumber(pdf_folder + "/" + pdf + ".pdf")
    #text = extract_text_with_tabs_and_spaces(pdf_folder + "/" + pdf + ".pdf",tab_threshold=30)
    #text = extract_clean_text(pdf_folder + "/" + pdf + ".pdf")
    print(text[i:i+100])
    print('\n')

    with open(txt_folder + "/" + pdf + ".txt", "w", encoding="utf-8") as f:
        f.write(text)

A

                                                          
                                        


DBDB

                                                          
                                        


DLMSLC

                                                          
                                        


GDC


KeyboardInterrupt: 